# Chương 5 — WCFR–XGBoost trên U.S. Accidents

Notebook chuyên sâu cho một pipeline: **WCFR–XGBoost**. Mục tiêu là phân loại Severity 1–4, mức ảnh hưởng tới giao thông trong dataset; không diễn giải nhãn như số người tử vong/bị thương. Notebook dùng cùng feature protocol và split seed với notebook tổng hợp để kết quả có thể đối chiếu.

## Lộ trình thực nghiệm

1. Khảo sát CSV và kiểm tra nhãn.
2. Xây feature hợp lệ tại thời điểm bắt đầu tai nạn, bỏ cột hậu nghiệm/leakage.
3. Tạo split nhóm xấp xỉ 64/16/20; giữ cùng seed giữa các notebook.
4. Fit preprocessing trên train, huấn luyện riêng mô hình trong notebook này.
5. Chọn cấu hình bằng validation; chỉ dùng test cho báo cáo cuối.
6. Lưu config, split IDs, metrics, dự đoán và biểu đồ vào thư mục run riêng.

## Nguồn và phạm vi thuật toán

Zhou, Wang & Zhang, Feature selection based on weighted conditional mutual information, Sections 3.1, Eqs. (11)–(17), especially Eq. (13), 2019/2020. https://doi.org/10.1016/j.aci.2019.12.003

Ghi chú paper và quyết định triển khai nằm trong `PAPER_IMPLEMENTATION_NOTES.md`. Cấu hình notebook là DEMO để kiểm tra pipeline, không tái lập benchmark của paper.

## Cấu hình và môi trường

Chạy tuần tự từ đầu. `MODE=FULL` là mặc định và dùng toàn bộ dòng sau khi bỏ duplicate chính xác; `DEMO` lấy tối đa 30.000 dòng phân tầng với seed 217 để kiểm tra nhanh. Bốn notebook được chạy riêng, từng notebook một. Có thể đặt `DATA_PATH` nếu auto-discovery tìm nhiều CSV cùng tên. XGBoost dùng tối đa 250 cây và early stopping; mạng neural bắt đầu với tối đa 8 epoch, batch 512 và early stopping theo validation. Các mức này là cấu hình full-data khởi điểm, không phải tuning benchmark. Chỉnh ngân sách trong CONFIG nếu cần. Kaggle cần Internet nếu package chính thức còn thiếu. Với FT-Transformer/TabM nên bật GPU; CPU vẫn được hỗ trợ nhưng có thể chậm.

In [ ]:
CONFIG = {
    "MODE": "FULL",                  # DEMO hoặc FULL
    "SEED": 217,
    "DATA_PATH": None,               # None để tự tìm Accidents_500.csv
    "DEMO_N": 30000,
    "RUN_MODELS": ["WCFR-XGBoost"],
    "WCFR_TOP_K": 10,                # cố định cho demo, không chọn trên test
    "WCFR_MAX_TRAIN_ROWS": 10000,
    "WCFR_NUMERIC_BINS": 5,
    "WCFR_MAX_CATEGORIES": 40,
    "MAX_WEATHER_AGE_MIN": 120,
    "MAX_CATEGORIES_PER_FEATURE": 128,
    "MIN_CATEGORY_COUNT": 2,
    "XGB_TREES": 250,
    "XGB_EARLY_STOPPING_ROUNDS": 20,
    "XGB_VERBOSE_EVAL": 25,
    "NEURAL_EPOCHS": 8,
    "NEURAL_PATIENCE": 3,
    "BATCH_SIZE": 512,
    "LEARNING_RATE": 0.0005,
    "WEIGHT_DECAY": 0.0001,
    "TABM_K": 8,
    "TRAIN_LOG_EVERY_STEPS": 100,
}
if CONFIG["MODE"] not in {"DEMO", "FULL"}:
    raise ValueError("MODE phải là DEMO hoặc FULL.")
if not CONFIG["RUN_MODELS"]:
    raise ValueError("RUN_MODELS cần có ít nhất một mô hình.")

import hashlib, importlib, importlib.metadata as metadata, json, os, platform
import random, subprocess, sys, time, warnings, shutil
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from scipy import sparse
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score,
    confusion_matrix, f1_score, precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
SEED = int(CONFIG["SEED"])
random.seed(SEED)
np.random.seed(SEED)
model_tag = "wcfr_xgboost"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + f"_{model_tag}_seed{SEED}"
IS_KAGGLE = Path("/kaggle/input").exists() and Path("/kaggle/working").exists()
OUTPUT_ROOT = Path("/kaggle/working/outputs") if IS_KAGGLE else Path.cwd() / "outputs"
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print("Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| sklearn", sklearn.__version__)
print("Kaggle:", IS_KAGGLE, "| CPU cores:", os.cpu_count())
print("Output:", OUTPUT_DIR.resolve())

### 5.2. Tìm CSV và khảo sát schema

Cell này chỉ tìm đúng tên Accidents_500.csv. Nếu nhiều bản cùng tên, cần đặt DATA_PATH thủ công. Thống kê bao gồm kích thước, kiểu, missing, duplicate, thời gian và nhãn. Checksum cho phép nhận diện đúng CSV đã dùng.

In [ ]:
def find_data_file(data_path):
    if data_path:
        path = Path(data_path).expanduser()
        if not path.is_file() or path.name.lower() != "accidents_500.csv":
            raise FileNotFoundError("DATA_PATH cần trỏ tới file Accidents_500.csv tồn tại.")
        return path.resolve()
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        matches = sorted(p for p in kaggle.rglob("Accidents_500.csv") if p.is_file())
        if len(matches) > 1:
            raise RuntimeError("Có nhiều CSV cùng tên; đặt CONFIG['DATA_PATH']:\n" +
                               "\n".join(map(str, matches)))
        if len(matches) == 1:
            return matches[0]
    local = Path.cwd() / "Accidents_500.csv"
    if local.is_file():
        return local.resolve()
    raise FileNotFoundError("Không thấy CSV. Add Input trên Kaggle hoặc đặt DATA_PATH.")

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

DATA_FILE = find_data_file(CONFIG["DATA_PATH"])
DATA_SHA256 = sha256_file(DATA_FILE)
df_raw = pd.read_csv(DATA_FILE, low_memory=False)
print("CSV:", DATA_FILE, "| bytes:", DATA_FILE.stat().st_size,
      "| SHA-256:", DATA_SHA256)
print("Shape:", df_raw.shape)
if "Severity" not in df_raw or df_raw["Severity"].isna().any():
    raise ValueError("Severity thiếu hoặc có giá trị missing.")
labels_seen = sorted(pd.to_numeric(df_raw["Severity"], errors="coerce").dropna().unique())
if labels_seen != [1, 2, 3, 4]:
    raise ValueError(f"Nhãn thực tế {labels_seen}; yêu cầu đủ Severity 1–4.")

schema = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": [str(df_raw[c].dtype) for c in df_raw.columns],
    "missing": [int(df_raw[c].isna().sum()) for c in df_raw.columns],
    "missing_pct": [float(df_raw[c].isna().mean() * 100) for c in df_raw.columns],
    "unique": [int(df_raw[c].nunique(dropna=True)) for c in df_raw.columns],
})
display(schema.sort_values("missing_pct", ascending=False))
label_counts = df_raw["Severity"].value_counts().sort_index()
print("Severity counts:", label_counts.to_dict())
print("Exact duplicate rows:", int(df_raw.duplicated().sum()))
print("Duplicate ID:", int(df_raw["ID"].duplicated().sum()) if "ID" in df_raw else "no ID")
for column in ["Start_Time", "End_Time", "Weather_Timestamp"]:
    if column in df_raw:
        try:
            parsed = pd.to_datetime(df_raw[column], format="mixed", errors="coerce")
        except TypeError:
            parsed = pd.to_datetime(df_raw[column], errors="coerce")
        print(column, "range:", parsed.min(), "to", parsed.max(),
              "| parse failures:", int(parsed.isna().sum()))
schema.to_csv(OUTPUT_DIR / "schema_missing_summary.csv", index=False)
pd.DataFrame({"Severity": label_counts.index, "count": label_counts.values}).to_csv(
    OUTPUT_DIR / "label_distribution_full.csv", index=False)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(label_counts.index.astype(str), label_counts.values, color="#3b82f6")
ax.set(title="Phân bố Severity trong CSV", xlabel="Severity (nhãn gốc)", ylabel="Số dòng")
ax.grid(axis="y", alpha=.25)
plt.show()

### 5.3. Feature và leakage

Thời điểm dự đoán được chốt tại lúc bắt đầu/ghi nhận sự cố. Giữ tọa độ bắt đầu, khu vực, thuộc tính đường, thời điểm bắt đầu đã chuyển thành biến chu kỳ và thời tiết chỉ khi Weather_Timestamp không sau Start_Time, độ trễ tối đa 120 phút.

Bỏ ID khỏi X nhưng lưu để truy vết; bỏ Source, Airport_Code, Street, Zipcode, Country (định danh/proxy, cardinality cao hoặc hằng); bỏ Severity khỏi X; bỏ End_Time, End_Lat, End_Lng, Description, Distance(mi), Weather_Timestamp vì hậu nghiệm/nguy cơ leakage. Không đưa cột khác vào X nếu chưa thẩm định.

CSV không có event ID riêng. Split sẽ nhóm các dòng theo Start_Time và tọa độ bắt đầu làm tròn 5 chữ số; đây là chữ ký gần đúng, có thể gộp tai nạn trùng thời gian/vị trí.

In [ ]:
NUMERIC_FEATURES = [
    "start_lat", "start_lng", "start_year", "hour_sin", "hour_cos",
    "weekday_sin", "weekday_cos", "month_sin", "month_cos", "is_weekend",
    "weather_age_minutes", "temperature_f", "wind_chill_f", "humidity_pct",
    "pressure_in", "visibility_mi", "wind_speed_mph", "precipitation_in",
]
CATEGORICAL_FEATURES = [
    "city", "county", "state", "timezone", "wind_direction", "weather_condition",
    "amenity", "bump", "crossing", "give_way", "junction", "no_exit", "railway",
    "roundabout", "station", "stop", "traffic_calming", "traffic_signal",
    "turning_loop", "sunrise_sunset", "civil_twilight", "nautical_twilight",
    "astronomical_twilight",
]
FEATURE_NAMES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET_TO_CODE = {1: 0, 2: 1, 3: 2, 4: 3}
CODE_TO_TARGET = {v: k for k, v in TARGET_TO_CODE.items()}
CLASS_CODES = np.arange(4, dtype=np.int64)
ROAD_MAP = {
    "amenity":"Amenity", "bump":"Bump", "crossing":"Crossing",
    "give_way":"Give_Way", "junction":"Junction", "no_exit":"No_Exit",
    "railway":"Railway", "roundabout":"Roundabout", "station":"Station",
    "stop":"Stop", "traffic_calming":"Traffic_Calming",
    "traffic_signal":"Traffic_Signal", "turning_loop":"Turning_Loop",
}
NUM_SOURCE = {
    "temperature_f":"Temperature(F)", "wind_chill_f":"Wind_Chill(F)",
    "humidity_pct":"Humidity(%)", "pressure_in":"Pressure(in)",
    "visibility_mi":"Visibility(mi)", "wind_speed_mph":"Wind_Speed(mph)",
    "precipitation_in":"Precipitation(in)",
}
CAT_SOURCE = {
    "city":"City", "county":"County", "state":"State", "timezone":"Timezone",
    "wind_direction":"Wind_Direction", "weather_condition":"Weather_Condition",
    "sunrise_sunset":"Sunrise_Sunset", "civil_twilight":"Civil_Twilight",
    "nautical_twilight":"Nautical_Twilight", "astronomical_twilight":"Astronomical_Twilight",
    **ROAD_MAP,
}

df = df_raw.copy()
df["_source_row"] = np.arange(len(df), dtype=np.int64)
df = df.drop_duplicates(subset=list(df_raw.columns), keep="first").reset_index(drop=True)

if CONFIG["MODE"] == "DEMO":
    if len(df) < CONFIG["DEMO_N"]:
        raise ValueError("DEMO_N lớn hơn số dòng còn lại.")
    sampled, _ = sklearn.model_selection.train_test_split(
        df, train_size=int(CONFIG["DEMO_N"]), random_state=SEED,
        stratify=df["Severity"].astype(int),
    )
    df_run = sampled.sort_values("_source_row").reset_index(drop=True)
else:
    df_run = df.reset_index(drop=True)

def make_safe_features(frame):
    try:
        start = pd.to_datetime(frame.get("Start_Time"), format="mixed", errors="coerce")
        weather = pd.to_datetime(
            frame.get("Weather_Timestamp", pd.Series(pd.NaT, index=frame.index)),
            format="mixed", errors="coerce",
        )
    except TypeError:
        start = pd.to_datetime(frame.get("Start_Time"), errors="coerce")
        weather = pd.to_datetime(
            frame.get("Weather_Timestamp", pd.Series(pd.NaT, index=frame.index)),
            errors="coerce",
        )
    age = (start - weather).dt.total_seconds() / 60
    weather_ok = age.ge(0) & age.le(float(CONFIG["MAX_WEATHER_AGE_MIN"]))
    hour = start.dt.hour + start.dt.minute.fillna(0) / 60
    weekday, month = start.dt.dayofweek, start.dt.month
    x = pd.DataFrame(index=frame.index)
    x["start_lat"] = pd.to_numeric(frame.get("Start_Lat"), errors="coerce")
    x["start_lng"] = pd.to_numeric(frame.get("Start_Lng"), errors="coerce")
    x["start_year"] = start.dt.year.astype(float)
    x["hour_sin"], x["hour_cos"] = np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24)
    x["weekday_sin"], x["weekday_cos"] = np.sin(2*np.pi*weekday/7), np.cos(2*np.pi*weekday/7)
    x["month_sin"], x["month_cos"] = np.sin(2*np.pi*(month-1)/12), np.cos(2*np.pi*(month-1)/12)
    x["is_weekend"] = weekday.isin([5, 6]).astype(float)
    x["weather_age_minutes"] = age.where(weather_ok)
    for out, source in NUM_SOURCE.items():
        x[out] = pd.to_numeric(frame[source], errors="coerce").where(weather_ok) if source in frame else np.nan
    for out, source in CAT_SOURCE.items():
        if source in frame:
            values = frame[source].astype("string").str.strip()
            if out in {"wind_direction", "weather_condition"}:
                values = values.where(weather_ok, pd.NA)
            x[out] = values
        else:
            x[out] = pd.Series(pd.NA, index=frame.index, dtype="string")
    for col in NUMERIC_FEATURES:
        x[col] = pd.to_numeric(x[col], errors="coerce")
    for col in CATEGORICAL_FEATURES:
        x[col] = x[col].astype("string").str.strip().replace("", pd.NA).fillna("__MISSING__").astype(str)
    return x[FEATURE_NAMES]

X = make_safe_features(df_run)
y_original = df_run["Severity"].astype(int).to_numpy()
y = np.array([TARGET_TO_CODE[int(v)] for v in y_original], dtype=np.int64)
assert "Severity" not in X.columns and not ({"ID", "End_Time", "Description"} & set(X.columns))
if set(np.unique(y)) != {0, 1, 2, 3}:
    raise ValueError("Sau DEMO sampling không còn đủ lớp; tăng DEMO_N hoặc dùng FULL.")
print("MODE:", CONFIG["MODE"], "| rows:", X.shape[0], "| features:", X.shape[1])
print("DEMO class counts:", pd.Series(y_original).value_counts().sort_index().to_dict())
display(X.head(3))

### 5.3. Split chung, tránh giao cắt nhóm

Dùng StratifiedGroupKFold hai lượt: fold ngoài xấp xỉ 20% làm test; một fold trong phần còn lại xấp xỉ 16% tổng làm validation; phần còn lại xấp xỉ 64% làm train. Split được dùng cho cả bốn phương pháp và ID được lưu. Lớp hiếm phải đạt ngưỡng tối thiểu trong từng tập, nếu không notebook dừng để tăng sample thay vì bỏ lớp.

In [ ]:
def event_groups(frame):
    try:
        start = pd.to_datetime(frame.get("Start_Time"), format="mixed", errors="coerce")
    except TypeError:
        start = pd.to_datetime(frame.get("Start_Time"), errors="coerce")
    lat, lng = pd.to_numeric(frame.get("Start_Lat"), errors="coerce"), pd.to_numeric(frame.get("Start_Lng"), errors="coerce")
    valid = start.notna() & lat.notna() & lng.notna()
    result = pd.Series(index=frame.index, dtype="string")
    result.loc[valid] = (
        start.loc[valid].dt.strftime("%Y-%m-%d %H:%M:%S") + "|"
        + lat.loc[valid].round(5).map(lambda z: f"{z:.5f}") + "|"
        + lng.loc[valid].round(5).map(lambda z: f"{z:.5f}")
    )
    fallback = frame.loc[~valid, "ID"].astype(str) if "ID" in frame else frame.loc[~valid, "_source_row"].astype(str)
    result.loc[~valid] = "unkeyed|" + fallback
    return result.astype(str).to_numpy()

groups = event_groups(df_run)
outer = StratifiedGroupKFold(5, shuffle=True, random_state=SEED)
trainval_pos, test_pos = next(outer.split(np.zeros(len(y)), y, groups))
inner = StratifiedGroupKFold(5, shuffle=True, random_state=SEED+1)
train_local, val_local = next(inner.split(
    np.zeros(len(trainval_pos)), y[trainval_pos], groups[trainval_pos]))
train_pos, val_pos = trainval_pos[train_local], trainval_pos[val_local]

def check_counts(name, values, minimum):
    counts = pd.Series(values).value_counts().reindex(CLASS_CODES, fill_value=0)
    readable = {CODE_TO_TARGET[i]: int(counts[i]) for i in CLASS_CODES}
    print(name, "n =", len(values), "| Severity =", readable)
    if (counts < minimum).any():
        raise ValueError(f"{name} có lớp dưới ngưỡng {minimum}: {readable}. Tăng DEMO_N hoặc chọn FULL.")

check_counts("Train", y[train_pos], 20)
check_counts("Validation", y[val_pos], 10)
check_counts("Test", y[test_pos], 10)
g_train, g_val, g_test = set(groups[train_pos]), set(groups[val_pos]), set(groups[test_pos])
assert not (g_train & g_val or g_train & g_test or g_val & g_test)

SPLIT = {
    "train": (X.iloc[train_pos].reset_index(drop=True), y[train_pos]),
    "validation": (X.iloc[val_pos].reset_index(drop=True), y[val_pos]),
    "test": (X.iloc[test_pos].reset_index(drop=True), y[test_pos]),
}
split_name = np.empty(len(df_run), dtype=object)
split_name[train_pos], split_name[val_pos], split_name[test_pos] = "train", "validation", "test"
pd.DataFrame({
    "split": split_name, "source_row_index": df_run["_source_row"].to_numpy(),
    "ID": df_run["ID"].to_numpy() if "ID" in df_run else df_run["_source_row"].to_numpy(),
    "Severity": y_original, "event_group_approx": groups,
}).to_csv(OUTPUT_DIR / "split_ids.csv", index=False)
with open(OUTPUT_DIR / "class_mapping.json", "w", encoding="utf-8") as f:
    json.dump({"original_to_encoded": {str(k):v for k,v in TARGET_TO_CODE.items()},
               "encoded_to_original": {str(k):v for k,v in CODE_TO_TARGET.items()}}, f, indent=2)
print("Split ratio:", {k: round(len(v[1])/len(y), 4) for k,v in SPLIT.items()})
group_table = pd.DataFrame({"group": groups, "severity": y_original}).groupby("group").agg(
    rows=("severity","size"), labels=("severity","nunique"))
print("Repeated approximate groups:", int((group_table.rows > 1).sum()),
      "| groups with multiple labels:", int((group_table.labels > 1).sum()))

## 5.4. Tiền xử lý riêng cho kiến trúc này

Median, vocabulary/category codes, discretization cần thiết và class weights chỉ được fit bằng train. XGBoost dùng sparse one-hot; mạng neural nhận numerical values đã scale cùng missing indicators và categorical integer codes. Cột unseen/rare đi vào mã 0. Các pipeline dưới đây dùng cùng feature hợp lệ và split.

In [ ]:
preprocessing_start = time.perf_counter()
X_train, y_train = SPLIT["train"]
X_val, y_val = SPLIT["validation"]
X_test, y_test = SPLIT["test"]

# Median, vocabulary, scaler và encoder chỉ fit trên train.
numeric_medians = X_train[NUMERIC_FEATURES].median().fillna(0.0)
n_num = len(NUMERIC_FEATURES)
def numeric_block(frame):
    values = frame[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").to_numpy(np.float32)
    missing = np.isnan(values).astype(np.float32)
    values = np.where(np.isnan(values), numeric_medians.to_numpy(np.float32)[None,:], values)
    return np.concatenate([values.astype(np.float32), missing], axis=1)

cat_vocabs, cat_cards = {}, []
for col in CATEGORICAL_FEATURES:
    counts = X_train[col].astype(str).value_counts()
    keep = [v for v,n in counts.items() if n >= CONFIG["MIN_CATEGORY_COUNT"] and v != "__MISSING__"]
    keep = sorted(keep, key=lambda v: (-int(counts[v]), str(v)))[:CONFIG["MAX_CATEGORIES_PER_FEATURE"]-2]
    vocab = {v:i+1 for i,v in enumerate(keep)}
    if (X_train[col].astype(str) == "__MISSING__").any():
        vocab["__MISSING__"] = len(vocab)+1
    cat_vocabs[col] = vocab
    cat_cards.append(len(vocab)+1)

def cat_block(frame):
    return np.column_stack([
        frame[col].astype(str).map(cat_vocabs[col]).fillna(0).astype(np.int64).to_numpy()
        for col in CATEGORICAL_FEATURES
    ]).astype(np.int64)

num_train, num_val, num_test = numeric_block(X_train), numeric_block(X_val), numeric_block(X_test)
train_cat, val_cat, test_cat = cat_block(X_train), cat_block(X_val), cat_block(X_test)
ohe_args = {
    "categories":[np.arange(n, dtype=np.int64) for n in cat_cards],
    "handle_unknown":"ignore", "dtype":np.float32,
}
try:
    ohe = OneHotEncoder(sparse_output=True, **ohe_args)
except TypeError:
    ohe = OneHotEncoder(sparse=True, **ohe_args)
train_cat_sparse = ohe.fit_transform(train_cat)
val_cat_sparse, test_cat_sparse = ohe.transform(val_cat), ohe.transform(test_cat)
X_train_xgb = sparse.hstack([sparse.csr_matrix(num_train), train_cat_sparse], format="csr")
X_val_xgb = sparse.hstack([sparse.csr_matrix(num_val), val_cat_sparse], format="csr")
X_test_xgb = sparse.hstack([sparse.csr_matrix(num_test), test_cat_sparse], format="csr")
print("Sparse CSR train:", X_train_xgb.shape, "| nnz:", X_train_xgb.nnz)
train_num_nn = val_num_nn = test_num_nn = None

class_counts = np.bincount(y_train, minlength=4)
if (class_counts == 0).any():
    raise ValueError("Train thiếu lớp.")
class_weights = len(y_train) / (4.0 * class_counts.astype(np.float64))
print("Class weights train:", {CODE_TO_TARGET[i]: round(float(class_weights[i]),4) for i in range(4)})
PREPROCESSING_SECONDS = time.perf_counter() - preprocessing_start
print("Preprocessing seconds:", round(PREPROCESSING_SECONDS, 2))


## 5.5. Metrics và kiểm soát output

Metric chính là Macro-F1; báo thêm Accuracy, Weighted-F1, Balanced Accuracy, QWK, metric từng lớp và confusion matrix với thứ tự nhãn cố định 1–4. QWK xét khoảng cách thứ tự nhãn nhưng Severity vẫn là mức ảnh hưởng giao thông. Validation dùng để chọn checkpoint/điểm dừng; test chỉ được chấm sau khi chọn xong. Không có score 0 giả cho nhánh lỗi.

In [ ]:
METRIC_COLS = ["run_id","mode","model","n_test","macro_f1","accuracy","weighted_f1",
               "balanced_accuracy","qwk","preprocessing_seconds","feature_selection_seconds",
               "training_seconds","inference_seconds"]
CLASS_COLS = ["run_id","mode","model","Severity","precision","recall","f1","support"]
metric_rows, class_rows, pred_frames, status_rows, histories, execution_devices = [], [], [], [], {}, {}
selected_features = []

def check_probabilities(prob, n):
    prob = np.asarray(prob, dtype=np.float64)
    if prob.shape != (n,4) or not np.isfinite(prob).all():
        raise ValueError(f"Xác suất cần shape ({n},4) và hữu hạn, nhận {prob.shape}.")
    if not np.allclose(prob.sum(axis=1), 1.0, atol=1e-4):
        raise ValueError("Tổng xác suất từng dòng không bằng 1.")
    return prob

def add_success(name, prob, training=0., inference=0., selection=0., preprocessing=0.):
    prob = check_probabilities(prob, len(y_test))
    pred = prob.argmax(1)
    metric_rows.append({
        "run_id":RUN_ID,"mode":CONFIG["MODE"],"model":name,"n_test":len(y_test),
        "macro_f1":f1_score(y_test,pred,labels=CLASS_CODES,average="macro",zero_division=0),
        "accuracy":accuracy_score(y_test,pred),
        "weighted_f1":f1_score(y_test,pred,labels=CLASS_CODES,average="weighted",zero_division=0),
        "balanced_accuracy":balanced_accuracy_score(y_test,pred),
        "qwk":cohen_kappa_score(y_test,pred,labels=CLASS_CODES,weights="quadratic"),
        "preprocessing_seconds":preprocessing,"feature_selection_seconds":selection,
        "training_seconds":training,"inference_seconds":inference})
    p,r,f,s = precision_recall_fscore_support(y_test,pred,labels=CLASS_CODES,zero_division=0)
    for i in range(4):
        class_rows.append({"run_id":RUN_ID,"mode":CONFIG["MODE"],"model":name,
                           "Severity":CODE_TO_TARGET[i],"precision":p[i],"recall":r[i],
                           "f1":f[i],"support":int(s[i])})
    out = pd.DataFrame({
        "run_id":RUN_ID,"mode":CONFIG["MODE"],"model":name,
        "ID":df_run.iloc[test_pos]["ID"].to_numpy() if "ID" in df_run else df_run.iloc[test_pos]["_source_row"].to_numpy(),
        "true_severity":[CODE_TO_TARGET[int(v)] for v in y_test],
        "predicted_severity":[CODE_TO_TARGET[int(v)] for v in pred]})
    for i in range(4): out[f"prob_severity_{i+1}"] = prob[:,i]
    pred_frames.append(out)
    status_rows.append({"run_id":RUN_ID,"mode":CONFIG["MODE"],"model":name,"status":"success","message":""})
    matrix = confusion_matrix(y_test,pred,labels=CLASS_CODES)
    fig, ax = plt.subplots(figsize=(5,4))
    image = ax.imshow(matrix,cmap="Blues")
    ax.set(title=f"Confusion — {name} ({CONFIG['MODE']})",xlabel="Dự đoán Severity",
           ylabel="Thực tế Severity",xticks=range(4),yticks=range(4),
           xticklabels=[1,2,3,4],yticklabels=[1,2,3,4])
    for row in range(4):
        for col in range(4):
            ax.text(col,row,str(matrix[row,col]),ha="center",va="center")
    fig.colorbar(image,ax=ax); fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"confusion_{name.lower().replace('-','_')}.png",dpi=140)
    plt.show()

def add_failure(name, error):
    status_rows.append({"run_id":RUN_ID,"mode":CONFIG["MODE"],"model":name,
                        "status":"failed","message":f"{type(error).__name__}: {error}"})
    print(name,"failed:",f"{type(error).__name__}: {error}")

def ensure_package(import_name, distribution):
    try:
        return importlib.import_module(import_name)
    except ImportError:
        print("Cài package còn thiếu:", distribution)
        subprocess.check_call([sys.executable,"-m","pip","install","--quiet",distribution])
        importlib.invalidate_caches()
        return importlib.import_module(import_name)

def package_version(distribution):
    try: return metadata.version(distribution)
    except metadata.PackageNotFoundError: return "not-installed"

def sync_torch():
    t = sys.modules.get("torch")
    if t is not None and t.cuda.is_available(): t.cuda.synchronize()

def release_model_memory():
    import gc
    globals().pop("model", None)
    globals().pop("logits", None)
    gc.collect()
    t = sys.modules.get("torch")
    if t is not None and t.cuda.is_available():
        t.cuda.empty_cache()

## 5.6. WCFR–XGBoost: filter chọn đặc trưng rồi phân loại

**Nguồn chính:** Zhou, Wang & Zhang, Sections 3.1, Eqs. (11)–(17), nhất là Eq. (13). WCFR là bộ chọn feature; XGBoost là bộ phân loại phía sau. Notebook này không đổi score WCFR thành Pearson, MI một biến, hay feature importance.

### Score WCFR gốc

Với target `Y`, feature ứng viên `f` và tập đã chọn `S`, định nghĩa score:

`J(f;S) = (1-delta_1) * sum_{s in S} I(f;Y|s) - (1-delta_2) * sum_{s in S} [I(f;s)-I(f;s|Y)]`.

`I(f;Y|s)` đo lượng thông tin về lớp mà `f` bổ sung khi đã biết `s`. Số hạng trong ngoặc thứ hai mô tả redundancy giữa hai feature sau khi xét lớp. `delta_1` và `delta_2` là độ lệch chuẩn tổng thể của các dãy số hạng relevance và redundancy tương ứng. Trong code, hai độ lệch chuẩn được tính trên các feature `s` hiện thuộc `S`; redundancy difference giữ nguyên dấu theo phương trình.

### Khởi tạo và chọn tuần tự

Khi `S` rỗng, nhóm dùng `argmax_f I(f;Y)` làm bước khởi tạo minh bạch; paper bàn đến bước khởi tạo trong pseudo-code nhưng không đặc tả đây là lựa chọn duy nhất cho CSV hỗn hợp này. Mỗi vòng sau đó tính `J(f;S)` cho mọi feature chưa chọn, thêm feature có score lớn nhất, rồi lặp đến Top-10. Đây là greedy forward selection, không đảm bảo tìm subset tối ưu toàn cục.

### Ước lượng cho dữ liệu này

MI/CMI tính từ entropy plug-in của bảng tần suất rời rạc, dùng log tự nhiên (đơn vị nat). Giá trị xác suất 0 bỏ khỏi tổng entropy; sai số âm rất nhỏ ở MI/CMI được clip về 0. Số liên tục chia tối đa 5 quantile bins với edges fit trên mẫu train của selector; duplicate edges gộp. Category hiếm gom mã 0, constant features bị bỏ. Do paper không quy định estimator rời rạc hóa cụ thể cho CSV này, đây là lựa chọn triển khai của nhóm.

Selector chỉ nhìn training rows (tối đa 10.000 dòng, stratified và seed cố định). Validation/test không tham gia fit selector. K cố định bằng 10 cho lần chạy này, không chọn theo test. Output lưu thứ tự, score, số dòng selector và thời gian chọn riêng.

### XGBoost phía sau và phân tích

Các feature được chọn theo đúng thứ tự WCFR sẽ được ánh xạ về cột tương ứng trong ma trận sparse one-hot, rồi đưa vào cùng cấu hình XGBoost đa lớp như nhánh full-feature trong notebook tổng hợp. Notebook độc lập chỉ chạy pipeline WCFR–XGBoost. Để đối chiếu định lượng với XGBoost đầy đủ trên cùng split/configuration, dùng notebook so sánh tổng hợp.

So sánh số feature và thời gian chọn với Macro-F1/test, per-class recall và confusion matrix. Điểm test thấp hơn/cao hơn không chứng minh WCFR phổ quát kém/tốt; chỉ mô tả tác động của Top-10, estimator và split demo này. Công thức bài Electronics (2026) khác công thức WCFR gốc và không được dùng trong code này.

In [ ]:
def run_xgb(name, train_matrix, val_matrix, test_matrix, selection_seconds=0.):
    xgb = ensure_package("xgboost","xgboost==3.4.1")
    ver = tuple(int(x.split("+")[0]) for x in xgb.__version__.split(".")[:2])
    use_gpu = False
    try:
        import torch
        use_gpu = torch.cuda.is_available()
    except ImportError:
        pass

    def build(gpu):
        params = {
            "objective":"multi:softprob","num_class":4,"n_estimators":int(CONFIG["XGB_TREES"]),
            "max_depth":6,"learning_rate":0.08,"subsample":0.8,"colsample_bytree":0.8,
            "reg_lambda":2.0,"eval_metric":"mlogloss","random_state":SEED,"n_jobs":-1,
        }
        if gpu:
            params.update({"tree_method":"hist","device":"cuda"} if ver >= (2,0) else {"tree_method":"gpu_hist"})
        else: params["tree_method"]="hist"
        if ver >= (1,6): params["early_stopping_rounds"]=int(CONFIG["XGB_EARLY_STOPPING_ROUNDS"])
        return xgb.XGBClassifier(**params)

    model = build(use_gpu)
    fit_args = {
        "sample_weight":class_weights[y_train],
        "eval_set":[(val_matrix,y_val)],
        "sample_weight_eval_set":[class_weights[y_val]],
        "verbose":int(CONFIG["XGB_VERBOSE_EVAL"]),
    }
    if ver < (1,6): fit_args["early_stopping_rounds"]=int(CONFIG["XGB_EARLY_STOPPING_ROUNDS"])
    training_device = "cuda" if use_gpu else "cpu"
    print(name,"fit rows",len(y_train),"trees max",CONFIG["XGB_TREES"],"device",training_device,flush=True)
    start = time.perf_counter()
    try:
        model.fit(train_matrix,y_train,**fit_args)
        sync_torch()
    except Exception as gpu_error:
        if not use_gpu: raise
        print("GPU fit lỗi, thử lại cùng cấu hình trên CPU:", repr(gpu_error))
        model=build(False); model.fit(train_matrix,y_train,**fit_args)
        training_device = "cpu"
    training=time.perf_counter()-start
    validation_curve=model.evals_result().get("validation_0",{}).get("mlogloss",[])
    if validation_curve:
        curve=pd.DataFrame({"iteration":np.arange(1,len(validation_curve)+1),
                            "validation_mlogloss":np.asarray(validation_curve,dtype=float)})
        curve.to_csv(OUTPUT_DIR/"xgboost_validation_curve.csv",index=False)
        print("Saved validation curve:",len(curve),"rounds",flush=True)
    inference_device = "cpu"
    if training_device == "cuda" and ver >= (2,0):
        # XGBoost sklearn inputs are SciPy sparse matrices in host memory.
        # Select CPU prediction explicitly to avoid its device-mismatch fallback.
        model.get_booster().set_param({"device":"cpu"})
    execution_devices[name] = {
        "training":training_device, "inference":inference_device,
        "input_format":"SciPy CSR on CPU", "batch_size":None,
    }
    model.predict_proba(test_matrix[:min(32,len(y_test))]); sync_torch()
    start=time.perf_counter()
    prob=model.predict_proba(test_matrix); sync_torch()
    inference=time.perf_counter()-start
    add_success(name,prob,training=training,inference=inference,selection=selection_seconds,
                preprocessing=PREPROCESSING_SECONDS)
    print(name,"xgboost",xgb.__version__,"training device",training_device,
          "inference device",inference_device,"best iteration",getattr(model,"best_iteration","n/a"))
    return model


In [ ]:
def entropy_codes(*columns):
    matrix=np.column_stack([np.asarray(c,dtype=np.int64) for c in columns])
    _,counts=np.unique(matrix,axis=0,return_counts=True)
    p=counts/counts.sum()
    return float(-np.sum(p[p>0]*np.log(p[p>0])))

def mi(a,b): return max(0.,entropy_codes(a)+entropy_codes(b)-entropy_codes(a,b))
def cmi(a,b,z): return max(0.,entropy_codes(a,z)+entropy_codes(b,z)-entropy_codes(z)-entropy_codes(a,b,z))

def fit_wcfr_discretizer(frame):
    out=pd.DataFrame(index=frame.index); rules={}
    for col in NUMERIC_FEATURES:
        x=pd.to_numeric(frame[col],errors="coerce").to_numpy(float); valid=np.isfinite(x)
        edges=np.unique(np.quantile(x[valid],np.linspace(0,1,CONFIG["WCFR_NUMERIC_BINS"]+1))) if valid.any() else np.array([])
        code=np.zeros(len(x),dtype=np.int64)
        if valid.any(): code[valid]=np.searchsorted(edges[1:-1],x[valid],side="right")+1
        out[col]=code; rules[col]={"type":"numeric","edges":edges}
    for col in CATEGORICAL_FEATURES:
        vals=frame[col].astype(str).to_numpy(); counts=Counter(vals)
        keep=sorted(counts,key=lambda z:(-counts[z],z))[:CONFIG["WCFR_MAX_CATEGORIES"]-1]
        vocab={v:i+1 for i,v in enumerate(keep)}
        out[col]=np.fromiter((vocab.get(v,0) for v in vals),dtype=np.int64,count=len(vals))
        rules[col]={"type":"categorical","vocab":vocab}
    return out[FEATURE_NAMES],rules

def wcfr_select(discrete, labels, top_k):
    candidates=[c for c in FEATURE_NAMES if discrete[c].nunique()>1]
    if len(candidates)<top_k: raise ValueError(f"Chỉ {len(candidates)} feature thay đổi, cần Top-{top_k}.")
    values={c:discrete[c].to_numpy(np.int64) for c in candidates}
    marginal={c:mi(values[c],labels) for c in candidates}
    first=max(candidates,key=lambda c:marginal[c])
    selected=[first]
    ranking=[{"rank":1,"feature":first,"selection_score":marginal[first],
              "criterion":"MI(feature; Severity), initialization"}]
    print("WCFR selection",f"1/{top_k}",first,"MI",round(marginal[first],6),flush=True)
    while len(selected)<top_k:
        best,best_score=None,-np.inf
        for f in candidates:
            if f in selected: continue
            relevance=[cmi(values[f],labels,values[s]) for s in selected]
            redundancy=[mi(values[f],values[s])-cmi(values[f],values[s],labels) for s in selected]
            score=(1-np.std(relevance,ddof=0))*sum(relevance)-(1-np.std(redundancy,ddof=0))*sum(redundancy)
            if score>best_score: best,best_score=f,float(score)
        if best is None or not np.isfinite(best_score): raise RuntimeError("WCFR không có score hữu hạn.")
        selected.append(best)
        ranking.append({"rank":len(selected),"feature":best,"selection_score":best_score,
                        "criterion":"Original WCFR Eq. (13)"})
        print("WCFR selection",f"{len(selected)}/{top_k}",best,"score",round(best_score,6),flush=True)
    if len(selected)!=len(set(selected)): raise AssertionError("WCFR có feature lặp.")
    return selected,pd.DataFrame(ranking)

def matrix_indices_by_feature():
    result={c:[i,n_num+i] for i,c in enumerate(NUMERIC_FEATURES)}
    offset=2*n_num
    for col,card in zip(CATEGORICAL_FEATURES,cat_cards):
        result[col]=list(range(offset,offset+card)); offset+=card
    return result

selected_features=[]
if "WCFR-XGBoost" in CONFIG["RUN_MODELS"]:
    try:
        start=time.perf_counter()
        positions=np.arange(len(X_train))
        if len(positions)>CONFIG["WCFR_MAX_TRAIN_ROWS"]:
            sampler=sklearn.model_selection.StratifiedShuffleSplit(
                n_splits=1,train_size=CONFIG["WCFR_MAX_TRAIN_ROWS"],random_state=SEED)
            positions,_=next(sampler.split(positions,y_train))
        wcfr_x,wcfr_rules=fit_wcfr_discretizer(X_train.iloc[positions].reset_index(drop=True))
        selected_features,ranking=wcfr_select(wcfr_x,y_train[positions],CONFIG["WCFR_TOP_K"])
        selection_time=time.perf_counter()-start
        ranking.insert(0,"run_id",RUN_ID); ranking.insert(1,"mode",CONFIG["MODE"])
        ranking["fit_rows"]=len(positions)
        ranking.to_csv(OUTPUT_DIR/"wcfr_selected_features.csv",index=False)
        matrix_map=matrix_indices_by_feature()
        indices=[i for c in selected_features for i in matrix_map[c]]
        print("WCFR selected order:",selected_features,"| fit rows:",len(positions))
        display(ranking)
        with open(OUTPUT_DIR/"wcfr_fit_config.json","w",encoding="utf-8") as f:
            json.dump({"paper_version":"Zhou et al. original WCFR, Eq. (13)",
                "fit_rows":len(positions),"seed":SEED,"top_k":CONFIG["WCFR_TOP_K"],
                "numeric_bins":CONFIG["WCFR_NUMERIC_BINS"],"selected_features":selected_features,
                "rules":{"n_quantile_edges":sum(len(r["edges"]) for r in wcfr_rules.values() if r["type"]=="numeric"),
                         "n_categories":sum(len(r["vocab"]) for r in wcfr_rules.values() if r["type"]=="categorical")}},f,indent=2)
        run_xgb(
            "WCFR-XGBoost",X_train_xgb[:,indices],X_val_xgb[:,indices],X_test_xgb[:,indices],
            selection_seconds=selection_time)
    except Exception as e: add_failure("WCFR-XGBoost",e)
    finally: release_model_memory()

## 5.7. Báo cáo và lưu run

Metric test chỉ được tạo nếu mô hình trả xác suất hữu hạn với shape `N_test × 4`, tổng xác suất mỗi dòng bằng 1. Output chứa trạng thái, dự đoán/xác suất, metric tổng hợp và theo lớp, confusion matrix, cấu hình, versions/hardware, split IDs và mapping Severity. Mạng neural có thêm learning curve. Với WCFR có ranking và fit configuration của selector.

In [ ]:
metrics=pd.DataFrame(metric_rows,columns=METRIC_COLS)
class_metrics=pd.DataFrame(class_rows,columns=CLASS_COLS)
statuses=pd.DataFrame(status_rows,columns=["run_id","mode","model","status","message"])
predictions=pd.concat(pred_frames,ignore_index=True) if pred_frames else pd.DataFrame(
    columns=["run_id","mode","model","ID","true_severity","predicted_severity",
             "prob_severity_1","prob_severity_2","prob_severity_3","prob_severity_4"])
metrics.to_csv(OUTPUT_DIR/"metrics.csv",index=False)
class_metrics.to_csv(OUTPUT_DIR/"class_metrics.csv",index=False)
statuses.to_csv(OUTPUT_DIR/"model_status.csv",index=False)
predictions.to_csv(OUTPUT_DIR/"predictions.csv",index=False)
if len(metrics):
    display(metrics.sort_values("macro_f1",ascending=False))
    display(class_metrics.sort_values(["model","Severity"]))
    view=metrics.sort_values("macro_f1")
    fig,ax=plt.subplots(figsize=(8,max(3.5,.65*len(view))))
    ax.barh(view.model,view.macro_f1,color="#0f766e")
    ax.set(title=f"Macro-F1 test — {CONFIG['MODE']}",xlabel="Macro-F1",xlim=(0,1))
    ax.grid(axis="x",alpha=.25); fig.tight_layout()
    fig.savefig(OUTPUT_DIR/"test_macro_f1.png",dpi=150); plt.show()
else:
    print("Chưa có mô hình tạo metric; xem model_status.csv.")
if histories:
    fig,axes=plt.subplots(len(histories),2,figsize=(10,3.5*len(histories)),squeeze=False)
    for row,(name,h) in enumerate(histories.items()):
        axes[row,0].plot(h.epoch,h.weighted_train_cross_entropy,marker="o")
        axes[row,0].set(title=name+" weighted train CE",xlabel="Epoch",ylabel="Loss")
        axes[row,1].plot(h.epoch,h.validation_macro_f1,marker="o",color="#be123c")
        axes[row,1].set(title=name+" validation Macro-F1",xlabel="Epoch",ylabel="Macro-F1")
    fig.tight_layout(); fig.savefig(OUTPUT_DIR/"learning_curves.png",dpi=150); plt.show()

with open(OUTPUT_DIR/"config.json","w",encoding="utf-8") as f:
    json.dump({**CONFIG,"run_id":RUN_ID,"data_file_name":DATA_FILE.name,
        "data_sha256":DATA_SHA256,"actual_rows_before_sample":len(df),
        "actual_rows_used":len(df_run),"features":FEATURE_NAMES,
        "split_rows":{k:len(v[1]) for k,v in SPLIT.items()},
        "class_weights_from_train":{str(CODE_TO_TARGET[i]):float(class_weights[i]) for i in range(4)},
        "wcfr_selected_features":selected_features},f,ensure_ascii=False,indent=2)
torch=sys.modules.get("torch")
gpu=bool(torch is not None and torch.cuda.is_available())
environment={"python":sys.version,"platform":platform.platform(),"processor":platform.processor(),
    "logical_cpu_count":os.cpu_count(),"gpu_available_for_torch":gpu,
    "gpu_name":torch.cuda.get_device_name(0) if gpu else None,
    "model_execution_devices":execution_devices,
    "packages":{"numpy":np.__version__,"pandas":pd.__version__,"scikit-learn":sklearn.__version__,
      "xgboost":package_version("xgboost"),"torch":package_version("torch"),
      "rtdl_revisiting_models":package_version("rtdl_revisiting_models"),"tabm":package_version("tabm")}}
with open(OUTPUT_DIR/"environment.json","w",encoding="utf-8") as f:
    json.dump(environment,f,ensure_ascii=False,indent=2)
display(statuses)

### Hạn chế và cách tiếp tục

Kết quả được gắn nhãn theo MODE trong config. Cả DEMO và FULL đều dùng split ngẫu nhiên có nhóm gần đúng, không phải đánh giá triển khai tương lai; temporal holdout là phân tích riêng. Weather chỉ được dùng khi timestamp không sau Start_Time và tuổi tối đa 120 phút—đây là giả định thời điểm dự đoán cần bảo vệ trong báo cáo. WCFR dùng plug-in MI/CMI và lượng hóa train-only do nhóm chọn. Không diễn giải một lần chạy thành kết luận thuật toán luôn tốt hơn.

Output được tạo tại `outputs/<run_id>` (local) hoặc `/kaggle/working/outputs/<run_id>` (Kaggle). Cell cuối tạo ZIP riêng cho mô hình này.

In [ ]:
zip_path=Path(shutil.make_archive(str(OUTPUT_ROOT/f"Chapter5_{model_tag}_{RUN_ID}"),"zip",root_dir=OUTPUT_DIR))
print("Output directory:",OUTPUT_DIR.resolve())
print("ZIP:",zip_path.resolve())
try:
    from IPython.display import FileLink, display as ipy_display
    ipy_display(FileLink(str(zip_path)))
except Exception:
    pass